# Movies Over Time Analysis

In [25]:
import pandas as pd
import matplotlib.pyplot as plt

In [26]:
df_movies = pd.read_csv("../data/movies.csv")

In [27]:
df_movies.head()

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000.0,46998772.0,Warner Bros.,146.0
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.8,65000.0,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000.0,58853106.0,Columbia Pictures,104.0
2,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.7,1200000.0,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000.0,538375067.0,Lucasfilm,124.0
3,Airplane!,PG,Comedy,1980,"July 2, 1980 (United States)",7.7,221000.0,Jim Abrahams,Jim Abrahams,Robert Hays,United States,3500000.0,83453539.0,Paramount Pictures,88.0
4,Caddyshack,R,Comedy,1980,"July 25, 1980 (United States)",7.3,108000.0,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,6000000.0,39846344.0,Orion Pictures,98.0


In [28]:
df_movies["released"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 7668 entries, 0 to 7667
Series name: released
Non-Null Count  Dtype 
--------------  ----- 
7666 non-null   object
dtypes: object(1)
memory usage: 60.0+ KB


### Preparing release date data

The `released` column contains both date information and additional text. To enable time-based analysis, the date part was extracted and converted into a proper datetime format.

In [29]:
df_movies["release_date_str"] = df_movies["released"].str.split("(").str[0].str.strip()
df_movies[["released", "release_date_str"]].head()

,released,release_date_str
0,"June 13, 1980 (United States)","June 13, 1980"
1,"July 2, 1980 (United States)","July 2, 1980"
2,"June 20, 1980 (United States)","June 20, 1980"
3,"July 2, 1980 (United States)","July 2, 1980"
4,"July 25, 1980 (United States)","July 25, 1980"


In [30]:
df_movies["release_date"] = pd.to_datetime(df_movies["release_date_str"], errors="coerce")

In [31]:
df_movies["release_year"] = df_movies["release_date"].dt.year

In [32]:
df_movies['release_year'].isna().sum()

np.int64(59)

Due to inconsistent date formats in the `released` column, the release year was extracted using a regular expression rather than relying solely on datetime conversion:

In [33]:
extracted_year = df_movies['released'].str.extract(r'(\d{4})')[0].astype(float)

In [34]:
df_movies['release_year'] = df_movies['release_year'].fillna(extracted_year)

In [35]:
df_movies['release_year'].isna().sum()

np.int64(2)

In [40]:
df_movies = df_movies.dropna(subset=['release_year'])


### Final Data Cleaning

In [44]:
columns = [
    'name',
    'release_year',
    'rating',
    'genre',
    'score',
    'votes'
]
df_movies = df_movies[columns]

In [46]:
df_movies = df_movies.dropna(
    subset=['release_year', 'rating', 'genre', 'score']
)

In [59]:
genre_counts = df_movies['genre'].value_counts()

valid_genres = genre_counts[genre_counts >= 10].index

df_movies = df_movies[df_movies['genre'].isin(valid_genres)]

In [62]:
rating_counts = df_movies['rating'].value_counts()

valid_ratings = rating_counts[rating_counts >= 10].index

df_movies = df_movies[df_movies['rating'].isin(valid_ratings)]

In [63]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7546 entries, 0 to 7659
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          7546 non-null   object 
 1   release_year  7546 non-null   float64
 2   rating        7546 non-null   object 
 3   genre         7546 non-null   object 
 4   score         7546 non-null   float64
 5   votes         7546 non-null   float64
dtypes: float64(3), object(3)
memory usage: 412.7+ KB
